# Predicción de Precios de Casas (House Prices)

Proyecto de regresión listo para **Google Colab**.

**Pregunta:** ¿podemos predecir el precio de venta de una casa a partir de sus características?

**Target:** `median_house_value`

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset clásico California Housing.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar datos

In [ ]:
url = "https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print("Nulos:")
print(df.isnull().sum())
df.describe().T

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["median_house_value"], kde=True, bins=50)
plt.title("Distribución de precios")
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
num_cols = df.select_dtypes(include=np.number).columns
sns.heatmap(df[num_cols].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlación")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="median_income", y="median_house_value", alpha=0.3, ax=axes[0])
axes[0].set_title("Ingreso mediano vs Precio")
sns.boxplot(data=df, x="ocean_proximity", y="median_house_value", ax=axes[1])
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_title("Proximidad al océano vs Precio")
plt.tight_layout()
plt.show()

### Hallazgos
- `median_income` correlaciona fuerte con el precio.
- Hay nulos en `total_bedrooms`.
- `ocean_proximity` es categórica importante.

## 3. Preparar datos

In [ ]:
X = df.drop(columns="median_house_value")
y = df["median_house_value"]

X = pd.get_dummies(X, columns=["ocean_proximity"], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos

In [ ]:
modelos = {
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    resultados.append({
        "modelo": nombre,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred, squared=False),
        "R2": r2_score(y_test, y_pred),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("R2", ascending=False)

## 5. Feature importance

In [ ]:
rf = pipes["Random Forest"].named_steps["model"]
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)

plt.figure(figsize=(8, 5))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Top features — Random Forest")
plt.tight_layout()
plt.show()

## 6. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap

rf_pipe = pipes["Random Forest"]
X_train_trans = rf_pipe.named_steps["imputer"].transform(X_train)
X_train_trans = rf_pipe.named_steps["scaler"].transform(X_train_trans)
model = rf_pipe.named_steps["model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_trans[:200])

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values, X_train_trans[:200], feature_names=X.columns, max_display=10, show=False)
plt.title("SHAP Summary — House Prices")
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values, X_train_trans[:200], feature_names=X.columns, plot_type="bar", max_display=10, show=False)
plt.title("Importancia media SHAP")
plt.tight_layout()
plt.show()

## 7. Conclusiones

- `median_income` es la variable más importante.
- R² suele estar en **0.75–0.85** en test con Random Forest o Gradient Boosting.
- Hay que imputar `total_bedrooms` y codificar `ocean_proximity`.

**Rango esperado:** R² ~0.78–0.84 | MAE ~40k–50k